# SeamlessM4T v2 Large → ~1B Compression
## S2ST-Focused: Minitron (NeurIPS 2024) + ShortGPT-BI (ACL 2025)

```
ARCHITECTURE RECAP (SeamlessM4T v2 Large  ≈ 2.3B params)
════════════════════════════════════════════════════════
speech_encoder   (w2v-BERT 2.0 Conformer, 24 layers)  ~635 M
text_encoder     (NLLB Transformer encoder, 24 layers) ~350 M   ← S2ST does NOT use this path
text_decoder     (NLLB Transformer decoder, 24 layers) ~850 M   ← shared; critical for S2ST
t2u_model        (NAR UnitY2, enc 6 + dec 6 layers)   ~260 M   ← CRITICAL for audio output
vocoder          (HiFi-GAN unit vocoder)                ~75 M   ← must be kept intact
embeddings/misc                                        ~130 M
────────────────────────────────────────────────────────
S2ST forward pass:  speech → speech_encoder → text_decoder → t2u_model → vocoder → waveform
text_encoder is NOT in the S2ST path, but its weights are shared with text_decoder cross-attn.

COMPRESSION PLAN  (2.3B → ~1.0B)  based on accepted literature
════════════════════════════════════════════════════════════════
Phase 0 : Baseline ASR-BLEU benchmark                      (reference)
Phase 1 : ShortGPT Block-Influence (BI) depth pruning      speech_encoder 24→12 layers  -~160M
Phase 2 : ShortGPT BI depth pruning                        text_decoder   24→12 layers  -~260M
Phase 3 : Minitron activation-based width pruning          FFN + MHA heads across enc+dec -~450M
Phase 4 : T2U depth pruning (BI)                           6→4 enc, 6→4 dec             -~90M
Phase 5 : LoRA S2ST fine-tuning (task-specific recovery)   quality restoration
Phase 6 : Final ASR-BLEU benchmark + paper table

ALGORITHMS USED (all from accepted, non-under-review venues)
  • ShortGPT Block Influence (BI)   → ACL Findings 2025 (Men et al.)
  • Minitron activation-importance  → NeurIPS 2024 (Muralidharan et al.)
  • LoRA fine-tuning                → ICLR 2022 (Hu et al.)
  • Knowledge Distillation (logit)  → NeurIPS 2015 (Hinton et al.)

METRIC: ASR-BLEU  (speech output → MMS-1B transcription → sacrebleu vs reference)
  This is the canonical SeamlessM4T S2ST evaluation metric used in the original paper.
  We do NOT use text BLEU / ChrF because the output is audio, not text.
  ASR backbone: facebook/mms-1b-all (replaces openai-whisper; MMS maintains >99% Bengali
  script fidelity vs Whisper large-v3 which collapses Bengali→Devanagari/Hindi).
```


## ── Setup Cells (run every session) ──

In [1]:
import os, sys, subprocess, pathlib, re, json, gc, copy, time, math, shutil, warnings
warnings.filterwarnings('ignore')

# Reduce CUDA memory fragmentation — set BEFORE any CUDA alloc
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# ── Platform detection ─────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = not ON_KAGGLE
PLATFORM  = 'kaggle' if ON_KAGGLE else 'colab'

# ── Path layout ───────────────────────────────────────────────────────────
GDRIVE_MOUNT = '/content/drive/MyDrive/cse465v7'
KAGGLE_WORK  = '/kaggle/working'
WORK_DIR  = KAGGLE_WORK   if ON_KAGGLE else GDRIVE_MOUNT
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
AUDIO_DIR = f'{WORK_DIR}/audio'
FIG_DIR   = f'{WORK_DIR}/figures'
MODEL_DIR = f'{WORK_DIR}/models'
GDRIVE_ROOT = 'gdrive:cse465v7'

for d in [WORK_DIR, CKPT_DIR, AUDIO_DIR, FIG_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Platform : {PLATFORM}')
print(f'Work dir : {WORK_DIR}')


Platform : kaggle
Work dir : /kaggle/working


In [2]:
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Drive mounted. Working folder: {GDRIVE_MOUNT}')
else:
    print('Kaggle: skipping Drive mount.')

Kaggle: skipping Drive mount.


In [3]:
if ON_KAGGLE:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash',
                   shell=True, capture_output=True)
    ver = subprocess.run('rclone version', shell=True, capture_output=True, text=True)
    print(ver.stdout.split('\n')[0])
else:
    print('Colab: rclone not needed.')

rclone v1.73.4


In [4]:
def _get_secret(key):
    """Fetch a secret from Kaggle Secrets or Colab userdata."""
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key)
        except Exception as e:
            raise RuntimeError(f'Kaggle secret {key!r} not found: {e}')
    else:
        try:
            from google.colab import userdata
            return userdata.get(key)
        except Exception as e:
            raise RuntimeError(
                f'Colab secret {key!r} not found. '
                f'Add it via the 🔑 Secrets panel in Colab: {e}')

if ON_KAGGLE:
    # rclone config is only needed on Kaggle
    RCLONE_CONF = _get_secret('RCLONE_CONF')
    raw = RCLONE_CONF.strip()
    raw = re.sub(r'\s*(\[[^\]]+\])\s*', r'\n\1\n', raw)
    raw = re.sub(r'\s+(type|scope|token|team_drive|client_id|client_secret|'
                 r'root_folder_id|service_account_file|drive_id)\s*=\s*',
                 r'\n\1 = ', raw)
    raw = raw.strip() + '\n'
    rclone_cfg = pathlib.Path.home() / '.config/rclone/rclone.conf'
    rclone_cfg.parent.mkdir(parents=True, exist_ok=True)
    rclone_cfg.write_text(raw)
    r = subprocess.run('rclone lsd gdrive:', shell=True, capture_output=True, text=True)
    print('Drive root:' if r.returncode == 0 else 'rclone FAILED:')
    print(r.stdout[:300] or r.stderr[:300])
else:
    print('Colab: skipping rclone config — using mounted Drive.')
    print(f'Working directory on Drive: {WORK_DIR}')

Drive root:
           0 2026-04-11 05:42:37        -1 .ipynb_checkpoints
           0 2026-04-17 11:03:10        -1 Colab Notebooks
           0 2025-11-10 11:33:43        -1 ScholarMate
           0 2026-04-05 12:59:09        -1 cse465
           0 2026-04-12 12:42:04        -1 cse465v5
           0 2026-04-1


In [5]:
# Install dependencies
subprocess.run(
    'pip install -q transformers>=4.40.0 datasets>=2.18.0 '
    'peft>=0.10.0 accelerate>=0.29.0 '
    'soundfile librosa sentencepiece sacrebleu jiwer '
    'torchaudio pyarrow',  # openai-whisper removed; using facebook/mms-1b-all instead
    shell=True, capture_output=True
)
print('Dependencies installed.')

Dependencies installed.


In [31]:
# ── Memory management utilities ────────────────────────────────────────────
import ctypes, contextlib

def _cuda_trim():
    """Ask CUDA to release all unneeded cached memory back to the OS."""
    if not torch.cuda.is_available():
        return
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    try:
        ctypes.CDLL('libcuda.so').cuMemTrim(0)
    except Exception:
        pass  # not available on all drivers — safe to ignore

def free_memory(*objs):
    """
    Fully deallocate model tensors and flush CUDA cache.
    Pass any number of model/tensor variables. Handles None safely.

    Example:
        free_memory(model_p1, work_model)
        model_p1 = None
    """
    import gc as _gc
    for obj in objs:
        if obj is None:
            continue
        try:
            if hasattr(obj, 'cpu'):
                obj.cpu()
        except Exception:
            pass
        try:
            del obj
        except Exception:
            pass
    _gc.collect()
    _cuda_trim()

def offload(model, name='model'):
    """Move model to CPU, free GPU cache, print VRAM. Returns model."""
    if model is None:
        return None
    model.cpu()
    gc.collect()
    _cuda_trim()
    print_vram(f'after offload {name}')
    return model

def print_vram(label=''):
    """Print current VRAM allocation."""
    if not torch.cuda.is_available():
        return
    alloc  = torch.cuda.memory_allocated()  / 1e9
    reserv = torch.cuda.memory_reserved()   / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    free   = total - reserv
    print(f'[VRAM{" "+label if label else ""}] alloc={alloc:.2f}GB  '
          f'reserv={reserv:.2f}GB  free≈{free:.2f}GB / {total:.2f}GB')

@contextlib.contextmanager
def cuda_phase(name):
    """
    Context manager that guarantees GPU memory is flushed even on exception.
    Usage:
        with cuda_phase('Phase 1 pruning'):
            ... do work ...
    """
    print_vram(f'[START] {name}')
    try:
        yield
    finally:
        gc.collect()
        _cuda_trim()
        print_vram(f'[END]   {name}')

print('Memory utilities ready.')


Memory utilities ready.


In [6]:
import torch
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import soundfile as sf
import torchaudio
from datasets import load_dataset, Dataset
from transformers import (
    AutoProcessor,
    SeamlessM4Tv2Model,
    SeamlessM4Tv2Config,
    Wav2Vec2ForCTC,
    AutoProcessor as MmsAutoProcessor,
)
from peft import LoraConfig, get_peft_model, TaskType
import sacrebleu

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'Device : {DEVICE}  dtype : {DTYPE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Device : cuda  dtype : torch.float16
GPU    : Tesla T4
VRAM   : 15.6 GB


In [7]:
# ── rclone helpers (Kaggle) / direct-path helpers (Colab) ─────────────────

def _rclone_pull(remote_rel, local_path, is_dir=False):
    """Download remote_rel from gdrive to local_path (Kaggle only)."""
    if not ON_KAGGLE:
        return
    cmd = (f'rclone copy "{GDRIVE_ROOT}/{remote_rel}" "{local_path}"'
           if is_dir else
           f'rclone copyto "{GDRIVE_ROOT}/{remote_rel}" "{local_path}"')
    subprocess.run(cmd, shell=True, capture_output=True)

def _rclone_push(local_path, remote_rel, is_dir=False):
    """Upload local_path to gdrive remote_rel (Kaggle only)."""
    if not ON_KAGGLE:
        return
    cmd = (f'rclone sync "{local_path}/" "{GDRIVE_ROOT}/{remote_rel}/"'
           if is_dir else
           f'rclone copyto "{local_path}" "{GDRIVE_ROOT}/{remote_rel}"')
    subprocess.run(cmd, shell=True, capture_output=True)

def save_model_to_drive(model, processor, name):
    """Save model+processor locally (and push to Drive on Kaggle)."""
    local = f'{MODEL_DIR}/{name}'
    os.makedirs(local, exist_ok=True)
    print(f"saving {local}")
    model.save_pretrained(local)
    processor.save_pretrained(local)
    _rclone_push(local, f'models/{name}', is_dir=True)
    print(f'[save] Model saved → {local}')

def load_model_from_drive(name):
    """Load model+processor from local (pull from Drive on Kaggle if missing)."""
    local = f'{MODEL_DIR}/{name}'
    if not os.path.exists(local):
        print(f'[load] {local} not found locally – pulling from Drive...')
        os.makedirs(local, exist_ok=True)
        _rclone_pull(f'models/{name}', local, is_dir=True)
    if not os.path.exists(f'{local}/config.json'):
        print(f'[load] Model {name} not found in Drive either. Return None.')
        return None, None
    print(f'[load] Loading {name} from {local}')
    model = SeamlessM4Tv2Model.from_pretrained(local, torch_dtype=DTYPE)
    processor = AutoProcessor.from_pretrained(local)
    return model, processor

def save_checkpoint(data, name, step=0):
    """Pickle-safe checkpoint (json for plain dicts)."""
    local = f'{CKPT_DIR}/{name}_step{step}.json'
    with open(local, 'w') as f:
        json.dump(data, f, default=str, indent=2)
    _rclone_push(local, f'checkpoints/{name}_step{step}.json')
    print(f'[ckpt] Saved → {local}')

def load_latest_checkpoint(name):
    """Load the most recent checkpoint matching <name>."""
    pattern = f'{CKPT_DIR}/{name}_step*.json'
    import glob
    files = sorted(glob.glob(pattern))
    if not files:
        # Try pulling from Drive (Kaggle)
        _rclone_pull(f'checkpoints/', CKPT_DIR, is_dir=True)
        files = sorted(glob.glob(pattern))
    if not files:
        return None
    path = files[-1]
    with open(path) as f:
        return json.load(f)

print('Drive helpers ready.')

Drive helpers ready.


In [9]:
# ── Model analysis utilities ───────────────────────────────────────────────

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

def print_model_breakdown(model, label='Model'):
    total = count_params(model)
    print(f'\n{"═"*60}')
    print(f'  {label}')
    print(f'  Total params: {total:.1f}M')
    print(f'  ─────────────────────────────────────────')
    components = [
        ('speech_encoder',  getattr(model, 'speech_encoder', None)),
        ('text_encoder',    getattr(model, 'text_encoder',   None)),
        ('text_decoder',    getattr(model, 'text_decoder',   None)),
        ('t2u_model',       getattr(model, 't2u_model',      None)),
        ('vocoder',         getattr(model, 'vocoder',        None)),
    ]
    for name, comp in components:
        if comp is not None:
            n = sum(p.numel() for p in comp.parameters()) / 1e6
            print(f'  {name:<20} {n:>8.1f}M  ({n/total*100:5.1f}%)')
    print(f'  {"─"*41}')
    # Layer counts
    cfg = model.config
    se_layers = getattr(cfg, 'speech_encoder_layers', '?')
    td_layers = getattr(cfg, 'decoder_layers', '?')
    t2u_enc   = getattr(cfg, 't2u_encoder_layers', '?')
    t2u_dec   = getattr(cfg, 't2u_decoder_layers', '?')
    print(f'  speech_enc layers   : {se_layers}')
    print(f'  text_dec layers     : {td_layers}')
    print(f'  t2u enc layers      : {t2u_enc}')
    print(f'  t2u dec layers      : {t2u_dec}')
    print(f'{"═"*60}')
    return total

def sync_config_from_model(model):
    """Sync config layer counts from actual layer lists after pruning."""
    cfg = model.config
    # Speech encoder
    se = getattr(model, 'speech_encoder', None)
    if se is not None:
        enc = getattr(se, 'encoder', None)
        if enc is not None and hasattr(enc, 'layers'):
            cfg.speech_encoder_layers = len(enc.layers)
    # Text decoder
    td = getattr(model, 'text_decoder', None)
    if td is not None and hasattr(td, 'layers'):
        cfg.decoder_layers = len(td.layers)
    # T2U encoder / decoder
    t2u = getattr(model, 't2u_model', None)
    if t2u is not None:
        t2u_m = getattr(t2u, 'model', None)
        t2u_enc = getattr(t2u_m, 'encoder', None) if t2u_m else None
        t2u_dec = getattr(t2u_m, 'decoder', None) if t2u_m else None
        if t2u_enc is not None and hasattr(t2u_enc, 'layers'):
            cfg.t2u_encoder_layers = len(t2u_enc.layers)
        if t2u_dec is not None and hasattr(t2u_dec, 'layers'):
            cfg.t2u_decoder_layers = len(t2u_dec.layers)

print('Model utilities ready.')

Model utilities ready.


## ── Dataset (Parquet-based FLEURS eng→ben) ──

In [10]:
import glob
import numpy as np
import torch
import torchaudio
import soundfile as sf
from datasets import load_dataset, Audio

SRC_LANG    = 'eng'
TGT_LANG    = 'ben'
SAMPLE_RATE = 16000

# ── Reduced sample counts for faster iteration ────────────────────────────
N_EVAL_SAMPLES  = 20   # was 50  — enough for stable BLEU estimate
N_CALIB_SAMPLES = 8    # was 32  — BI calibration; 8 fwd passes is sufficient

def load_fleurs_parquet(split='test', max_samples=20):
    en_pattern = f'/kaggle/input/**/en_us/*{split}*.parquet'
    bn_pattern = f'/kaggle/input/**/bn_in/*{split}*.parquet'
    en_files = glob.glob(en_pattern, recursive=True)
    bn_files = glob.glob(bn_pattern, recursive=True)
    samples = []

    if en_files and bn_files:
        print(f"Found local parquet files for '{split}' split.")
        en_ds = load_dataset('parquet', data_files=en_files, split='train')
        bn_ds = load_dataset('parquet', data_files=bn_files, split='train')
        en_ds = en_ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

        bn_text_map = {}
        for row in bn_ds:
            if 'id' in row:
                for field in ['transcription', 'raw_transcription', 'sentence',
                               'target_text', 'text']:
                    if field in row and row[field]:
                        bn_text_map[row['id']] = str(row[field]).strip()
                        break

        for row in en_ds:
            if len(samples) >= max_samples:
                break
            row_id = row.get('id')
            if row_id is None or row_id not in bn_text_map:
                continue
            ref_text = bn_text_map[row_id]
            if not ref_text:
                continue
            audio_col = row.get('audio')
            arr = np.array(audio_col['array'], dtype=np.float32)
            sr  = int(audio_col['sampling_rate'])
            if sr != SAMPLE_RATE:
                t = torch.tensor(arr).unsqueeze(0)
                t = torchaudio.functional.resample(t, sr, SAMPLE_RATE)
                arr = t.squeeze(0).numpy()
            samples.append({'audio_array': arr, 'sr': SAMPLE_RATE,
                            'reference_text': ref_text})

    print(f'Loaded {len(samples)} samples ({split} split).')
    return samples

eval_samples  = load_fleurs_parquet(split='test',  max_samples=N_EVAL_SAMPLES)
calib_samples = load_fleurs_parquet(split='train', max_samples=N_CALIB_SAMPLES)


Found local parquet files for 'test' split.
EN files: 1 | BN files: 2


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 50 samples (test split).
Found local parquet files for 'train' split.
EN files: 4 | BN files: 5


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 32 samples (train split).


## ── ASR-BLEU Evaluation (canonical S2ST metric) ──

In [11]:
# ── MMS-1B ASR backbone + evaluation helpers ─────────────────────────────
from IPython.display import Audio as IPAudio, display
from tqdm import tqdm
import time

MMS_MODEL_ID = 'facebook/mms-1b-all'
SEAMLESS_TO_MMS_LANG = {'ben': 'ben', 'hin': 'hin', 'eng': 'eng'}

_mms_model     = None
_mms_processor = None
_mms_lang      = None

def get_mms(language='ben'):
    """Lazy-load MMS-1B and hot-swap language adapter."""
    global _mms_model, _mms_processor, _mms_lang
    if _mms_model is None:
        print(f'Loading {MMS_MODEL_ID} backbone...')
        _mms_processor = MmsAutoProcessor.from_pretrained(
            MMS_MODEL_ID, target_lang=language)
        _mms_model = Wav2Vec2ForCTC.from_pretrained(
            MMS_MODEL_ID, target_lang=language,
            ignore_mismatched_sizes=True, torch_dtype=DTYPE,
        ).to(DEVICE)
        _mms_model.eval()
        _mms_lang = language
        print(f'  MMS-1B loaded | adapter: {language}')
    elif _mms_lang != language:
        _mms_processor.tokenizer.set_target_lang(language)
        _mms_model.load_adapter(language)
        _mms_lang = language
    return _mms_model, _mms_processor

def free_mms():
    """Unload MMS from GPU — call before heavy pruning phases."""
    global _mms_model, _mms_processor, _mms_lang
    if _mms_model is not None:
        free_memory(_mms_model)
        _mms_model = None
        _mms_processor = None
        _mms_lang = None
        print('MMS-1B freed from GPU.')

@torch.no_grad()
def transcribe_audio(audio_array, language='bn'):
    """Transcribe numpy audio (float32, 16 kHz) with MMS-1B."""
    lang_map = {'bn': 'ben', 'hi': 'hin', 'en': 'eng'}
    lang_iso = lang_map.get(language, language)
    model, processor = get_mms(lang_iso)
    arr = audio_array.astype(np.float32)
    inputs = processor(arr, sampling_rate=SAMPLE_RATE, return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.autocast(DEVICE, dtype=DTYPE):
        logits = model(**inputs).logits
    ids = torch.argmax(logits, dim=-1)
    return processor.decode(ids[0]).strip()

def compute_asr_bleu(hypotheses, references):
    """Corpus-level sacreBLEU."""
    return sacrebleu.corpus_bleu(hypotheses, [references]).score

def compute_sentence_bleu(hyp, ref):
    """Single-sentence sacreBLEU (for live display)."""
    try:
        return sacrebleu.sentence_bleu(hyp, [ref]).score
    except Exception:
        return 0.0

@torch.no_grad()
def run_s2st_inference(model, processor, audio_array,
                       tgt_lang='ben', src_lang='eng', max_new_tokens=256):
    """
    Run S2ST on one audio sample.
    max_new_tokens reduced to 256 (was 512) to save memory.
    Returns (waveform_np, gen_text, rtf)
    """
    model.eval()
    inputs = processor(
        audio=audio_array, src_lang=src_lang,
        sampling_rate=SAMPLE_RATE, return_tensors='pt'
    ).to(DEVICE)

    t0 = time.time()
    with torch.autocast(DEVICE, dtype=DTYPE):
        out = model.generate(
            **inputs, tgt_lang=tgt_lang,
            return_intermediate_token_ids=True,
            max_new_tokens=max_new_tokens,
        )
    elapsed = time.time() - t0

    waveform = out[0].cpu().squeeze().float().numpy()
    text_ids = out[1][0] if (isinstance(out, (list, tuple)) and len(out) > 1) else None
    gen_text = ''
    if text_ids is not None:
        try:
            gen_text = processor.decode(text_ids, skip_special_tokens=True)
        except Exception:
            pass

    # Explicitly delete GPU tensors
    del inputs, out
    torch.cuda.empty_cache()

    rtf = elapsed / max(len(audio_array) / SAMPLE_RATE, 1e-6)
    return waveform, gen_text, rtf


def benchmark_asr_bleu(model, processor, samples, label='',
                        save_n=2, preview_n=3):
    """
    Full ASR-BLEU benchmark.
    • Prints per-sample sentence BLEU live.
    • Frees MMS from GPU while SeamlessM4T runs, reloads for ASR, then frees again.
    • Uses cuda_phase context manager to guarantee cleanup on exception.
    Returns (hypotheses, references, summary_dict)
    """
    print(f'\n🚀  benchmark_asr_bleu({label})  — {len(samples)} samples')
    print_vram('benchmark start')

    hypotheses, references, rtfs, sent_bleus = [], [], [], []
    asr_lang = SEAMLESS_TO_MMS_LANG.get(TGT_LANG, TGT_LANG)
    start_time = time.time()

    # Free MMS while SeamlessM4T runs (both are ~4 GB — can't coexist on 15 GB)
    free_mms()
    model.eval()

    for i, s in enumerate(tqdm(samples, desc=label)):
        waveform, gen_text, rtf = None, '', 0.0
        asr_text = ''
        try:
            with cuda_phase(f'sample {i} s2st'):
                waveform, gen_text, rtf = run_s2st_inference(
                    model, processor, s['audio_array'],
                    tgt_lang=TGT_LANG, src_lang=SRC_LANG
                )

            # Load MMS only for this one ASR call, then flush
            asr_text = transcribe_audio(waveform, language=asr_lang)
            free_mms()   # free ~4 GB right after each transcription

            s_bleu = compute_sentence_bleu(asr_text, s['reference_text'])
            sent_bleus.append(s_bleu)
            hypotheses.append(asr_text)
            references.append(s['reference_text'])
            rtfs.append(rtf)

            if i < save_n and waveform is not None:
                sf.write(f'{AUDIO_DIR}/{label}_sample{i}.wav', waveform, SAMPLE_RATE)

            if i < preview_n and waveform is not None:
                print(f'\n🎧 Sample {i+1}')
                print('Input:')
                display(IPAudio(s['audio_array'], rate=SAMPLE_RATE))
                print('Generated:')
                display(IPAudio(waveform, rate=SAMPLE_RATE))
                print(f'  📝 Ref      : {s["reference_text"]}')
                print(f'  🤖 ASR      : {asr_text}')
                print(f'  📊 Sent-BLEU: {s_bleu:.2f}  |  RTF: {rtf:.3f}')
            else:
                print(f'  [{i+1:02d}] Sent-BLEU={s_bleu:.2f}  RTF={rtf:.3f}  '
                      f'asr={asr_text[:60]}')

        except Exception as e:
            print(f'\n⚠️  sample {i} failed: {e}')
            hypotheses.append('')
            references.append(s['reference_text'])
            rtfs.append(0.0)
            sent_bleus.append(0.0)
        finally:
            # Always clean up waveform to avoid accumulation
            del waveform
            gc.collect()
            torch.cuda.empty_cache()

        if (i + 1) % 5 == 0:
            corpus_so_far = compute_asr_bleu(hypotheses, references)
            elapsed = time.time() - start_time
            print(f'  ⏳ {i+1}/{len(samples)} | corpus BLEU so far={corpus_so_far:.2f} '
                  f'| {elapsed/(i+1):.1f}s/sample')

    bleu    = compute_asr_bleu(hypotheses, references)
    avg_rtf = float(np.mean([r for r in rtfs if r > 0]) if any(r > 0 for r in rtfs) else 0.0)
    params_m = count_params(model)

    summary = dict(label=label, params_M=params_m, asr_bleu=bleu,
                   avg_rtf=avg_rtf, n_samples=len(samples))

    print(f'\n✅ ── {label} ──────────────────────────────────')
    print(f'  📦 Params        : {params_m:.1f}M')
    print(f'  🎯 Corpus BLEU   : {bleu:.2f}')
    print(f'  📊 Avg Sent-BLEU : {np.mean(sent_bleus):.2f}  '
          f'(min={min(sent_bleus):.2f}, max={max(sent_bleus):.2f})')
    print(f'  ⚡ Avg RTF       : {avg_rtf:.4f}')
    print('─'*50)

    return hypotheses, references, summary

print('Evaluation functions ready.  ASR backbone: facebook/mms-1b-all')


Evaluation functions ready. ASR backbone: facebook/mms-1b-all


In [12]:
# ── Global summary tracker ─────────────────────────────────────────────────
ALL_SUMMARIES = []

def store_summary(s):
    ALL_SUMMARIES.append(s)
    save_checkpoint({'summaries': ALL_SUMMARIES}, 'all_summaries', step=len(ALL_SUMMARIES))

# Restore if session restarted
sc = load_latest_checkpoint('all_summaries')
if sc and 'summaries' in sc:
    ALL_SUMMARIES = sc['summaries']
    print(f'Restored {len(ALL_SUMMARIES)} phase summaries.')
else:
    print('No previous summaries found – starting fresh.')

No previous summaries found – starting fresh.


---
# Phase 0 — Baseline Benchmark
Load base `facebook/seamless-m4t-v2-large` and measure ASR-BLEU.
All subsequent phases will compare against this reference.


In [13]:
# ── P0 Cell 1: Load base model ───────────────────────────────────────────
with cuda_phase('load base model'):
    base_model, base_processor = load_model_from_drive('phase0_baseline')
    if base_model is None:
        print('Downloading facebook/seamless-m4t-v2-large ...')
        base_processor = AutoProcessor.from_pretrained('facebook/seamless-m4t-v2-large')
        base_model = SeamlessM4Tv2Model.from_pretrained(
            'facebook/seamless-m4t-v2-large', torch_dtype=DTYPE)
        save_model_to_drive(base_model, base_processor, 'phase0_baseline')
    base_model = base_model.to(DEVICE)
    print_model_breakdown(base_model, 'Phase 0 — Baseline')


[load] /kaggle/working/models/phase0_baseline not found locally – pulling from Drive...
[load] Loading phase0_baseline from /kaggle/working/models/phase0_baseline


Instantiating a decoder SeamlessM4Tv2Attention without passing `layer_idx` is not recommended and will lead to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


Loading weights:   0%|          | 0/2232 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
  Phase 0 — Baseline
  Total params: 2309.2M
  ─────────────────────────────────────────
  speech_encoder          635.0M  ( 27.5%)
  text_encoder            766.0M  ( 33.2%)
  text_decoder            866.8M  ( 37.5%)
  t2u_model               261.8M  ( 11.3%)
  vocoder                  41.9M  (  1.8%)
  ─────────────────────────────────────────
  speech_enc layers   : 24
  text_dec layers     : 24
  t2u enc layers      : 6
  t2u dec layers      : 6
════════════════════════════════════════════════════════════


2309.249669

In [14]:
# ── P0 Cell 2: Baseline ASR-BLEU ─────────────────────────────────────────
p0_ckpt = load_latest_checkpoint('phase0_benchmark')
if p0_ckpt:
    p0_summary = p0_ckpt
    print(f'Restored P0: ASR-BLEU={p0_summary["asr_bleu"]:.2f}')
else:
    free_mms()   # ensure MMS is not eating GPU while SeamlessM4T runs
    _, _, p0_summary = benchmark_asr_bleu(
        base_model, base_processor, eval_samples, label='P0_Baseline')
    save_checkpoint(p0_summary, 'phase0_benchmark')

store_summary(p0_summary)



🚀 Running benchmark_asr_bleu(P0_Baseline)


Processing samples:   0%|          | 0/50 [00:00<?, ?it/s]

Loading facebook/mms-1b-all backbone...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Processing samples:  90%|█████████ | 45/50 [02:20<00:11,  2.29s/it]


⏳ 45/50 done | Avg/sample: 3.11s


Processing samples: 100%|██████████| 50/50 [02:32<00:00,  3.04s/it]


⏳ 50/50 done | Avg/sample: 3.04s

✅ ── P0_Baseline COMPLETE ──
📦 Params   : 2309.2M
🎯 ASR-BLEU : 10.67  (via MMS-1B, lang=ben)
⚡ Avg RTF  : 0.2603


[ckpt] Saved → /kaggle/working/checkpoints/phase0_benchmark_step0.json
[ckpt] Saved → /kaggle/working/checkpoints/all_summaries_step1.json


In [32]:
# Offload base model to CPU — keep it alive for BI calibration
base_model = offload(base_model, 'base_model')
print('Base model on CPU.')


[VRAM after offloading p1] allocated=15.29GB  reserved=15.48GB  free≈0.16GB / 15.64GB
Base model moved to CPU.


---
# Pruning Algorithm Reference

## Algorithm 1 — ShortGPT Block Influence (BI)
**Source:** Men et al., *ShortGPT: Layers in Large Language Models are More Redundant Than You Expect*, ACL Findings 2025.

For each transformer block $i$, the Block Influence score is:
$$\text{BI}_i = 1 - \mathbb{E}_{X}\left[\cos(\mathbf{h}_i^{\text{in}}, \mathbf{h}_i^{\text{out}})\right]$$
where $\mathbf{h}_i^{\text{in}}$ and $\mathbf{h}_i^{\text{out}}$ are the input/output hidden states of block $i$.  
A block with BI ≈ 0 makes nearly no change to its input → safe to remove.

**We use this for: speech_encoder layers, text_decoder layers, t2u layers.**

## Algorithm 2 — Minitron Activation-Based Width Importance
**Source:** Muralidharan et al., *Compact Language Models via Pruning and Knowledge Distillation*, NeurIPS 2024.

For FFN hidden neurons, the importance of neuron $j$ in layer $l$ is:
$$I_j^l = \mathbb{E}_{X}\left[|\mathbf{a}_j^l|\right]$$
where $\mathbf{a}_j^l$ is the post-activation value. For attention heads, importance is the mean activation norm of head outputs. We rank and prune the least important neurons/heads while **reconstructing** the remaining weight rows via least-squares to preserve output statistics.

**We use this for: FFN intermediate dims and attention heads in text_decoder.**

## Algorithm 3 — LoRA Fine-Tuning for Recovery
**Source:** Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR 2022.

We inject low-rank adapters ($r=16$, $\alpha=32$) into the Q/K/V/O attention projections of the text_decoder and t2u_model. Only adapter weights are trained. Loss = cross-entropy on target text tokens (S2TT supervision), which propagates quality signal back through the entire S2ST pipeline.


---
# Phase 1 — Speech Encoder Depth Pruning (ShortGPT BI)
Target: 24 → 12 Conformer layers  (~−160M params)


In [34]:
# ── ShortGPT Block Influence implementation ─────────────────────────────────

@torch.no_grad()
def compute_bi_scores(layers_list, calibration_inputs_fn,
                      n_samples=32, device='cuda', fp16=True):
    """
    Compute Block Influence scores for a list of transformer layers.
    `calibration_inputs_fn(device)` → generator of hidden state tensors (B, T, D).
    Returns: list of BI scores, one per layer.
    """
    n_layers = len(layers_list)
    bi_scores = [0.0] * n_layers
    cos = torch.nn.CosineSimilarity(dim=-1)

    # We accumulate running stats: cos-sim per layer
    layer_cos_sums = [0.0] * n_layers
    layer_counts   = [0]   * n_layers

    # Forward hooks to capture input/output of each layer
    hooks, h_in, h_out = [], {}, {}

    def make_hook_in(idx):
        def hook(module, inp, out):
            h_in[idx] = inp[0].detach().float()
        return hook

    def make_hook_out(idx):
        def hook(module, inp, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            h_out[idx] = o.detach().float()
        return hook

    for i, layer in enumerate(layers_list):
        hooks.append(layer.register_forward_hook(make_hook_in(i)))
        hooks.append(layer.register_forward_hook(make_hook_out(i)))

    # PROBLEM: forward hooks get both in+out from the SAME hook.
    # Use pre-hooks for input.
    for h in hooks:
        h.remove()
    hooks = []
    h_in, h_out = {}, {}

    def make_pre_hook(idx):
        def hook(module, inp):
            h_in[idx] = inp[0].detach().float() if isinstance(inp, tuple) else inp.detach().float()
        return hook

    def make_post_hook(idx):
        def hook(module, inp, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            h_out[idx] = o.detach().float()
        return hook

    for i, layer in enumerate(layers_list):
        hooks.append(layer.register_forward_pre_hook(make_pre_hook(i)))
        hooks.append(layer.register_forward_hook(make_post_hook(i)))

    for batch in calibration_inputs_fn(device):
        h_in.clear(); h_out.clear()
        try:
            # Push the batch through the parent model—hooks will fire
            # `batch` is a dict of model inputs (already on device)
            batch
            # The hooks will collect data on the NEXT forward pass.
            # We need to trigger a partial forward. We do that outside this fn.
            break
        except Exception:
            break

    for h in hooks:
        h.remove()

    return bi_scores  # computed in the caller after triggering forward passes


# Better: integrated BI scorer that runs its own forward passes.
# ── ShortGPT Block Influence — fixed for variable-length conformer inputs ──

@torch.no_grad()
def compute_bi_for_submodule(parent_model, layers_attr_path, calib_fn,
                              n_calib=16, dtype=torch.float16):
    """
    Compute BI scores by registering hooks on a specific sub-module layer list.

    `layers_attr_path` : dotted path e.g. 'speech_encoder.encoder.layers'
    `calib_fn(model, device)` : function that runs a forward pass through parent_model
                                 using calibration data (no return value needed).

    Returns list[float] of BI scores, length = number of layers.

    Fix: mean-pool over time axis before storing hidden states so that
    variable-length sequences (different padding per sample) don't cause
    torch.cat to fail with size-mismatch errors.
    """
    import functools
    layers = functools.reduce(getattr, layers_attr_path.split('.'), parent_model)
    n      = len(layers)

    h_in_store  = {i: [] for i in range(n)}
    h_out_store = {i: [] for i in range(n)}

    def make_pre(i):
        def hook(mod, inp):
            x = inp[0] if isinstance(inp, tuple) else inp
            # Mean-pool over time so shape is (1, D) — safe to cat across samples
            h_in_store[i].append(x.detach().float().mean(dim=1, keepdim=True).cpu())
        return hook

    def make_post(i):
        def hook(mod, inp, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            # Mean-pool over time so shape is (1, D) — safe to cat across samples
            h_out_store[i].append(o.detach().float().mean(dim=1, keepdim=True).cpu())
        return hook

    hooks = []
    for i, layer in enumerate(layers):
        hooks.append(layer.register_forward_pre_hook(make_pre(i)))
        hooks.append(layer.register_forward_hook(make_post(i)))

    parent_model.eval()
    dev = next(parent_model.parameters()).device

    for _ in range(n_calib):
        try:
            calib_fn(parent_model, dev)
        except Exception as e:
            print(f'  [BI calib warning] {e}')

    for h in hooks:
        h.remove()

    cos = torch.nn.CosineSimilarity(dim=-1)
    bi_scores = []
    for i in range(n):
        if not h_in_store[i] or not h_out_store[i]:
            bi_scores.append(1.0)   # penalise if no data — keep layer
            continue
        inp_cat = torch.cat(h_in_store[i],  dim=0).float()  # (N, 1, D)
        out_cat = torch.cat(h_out_store[i], dim=0).float()  # (N, 1, D)
        # BI = 1 - mean cosine similarity
        sim = cos(inp_cat.reshape(-1, inp_cat.size(-1)),
                  out_cat.reshape(-1, out_cat.size(-1))).mean().item()
        bi_scores.append(1.0 - sim)

    return bi_scores


def select_layers_to_keep(bi_scores, keep_n, always_keep_first=1, always_keep_last=1):
    """
    Select `keep_n` layers with highest BI scores.
    Always keep the first `always_keep_first` and last `always_keep_last` layers
    (they usually handle embedding in/out bridging).
    Returns sorted list of indices to keep.
    """
    n = len(bi_scores)
    forced = set(range(always_keep_first)) | set(range(n - always_keep_last, n))
    candidates = [i for i in range(n) if i not in forced]
    remaining  = keep_n - len(forced)
    remaining  = max(0, remaining)
    # Sort candidates by BI descending, take top `remaining`
    sorted_cands = sorted(candidates, key=lambda i: bi_scores[i], reverse=True)
    selected = list(forced) + sorted_cands[:remaining]
    return sorted(selected)


def prune_layers(layer_list, keep_indices):
    """Return a new nn.ModuleList with only the kept layers (in order)."""
    import torch.nn as nn
    kept = nn.ModuleList([layer_list[i] for i in keep_indices])
    return kept

print('ShortGPT BI functions defined.')

ShortGPT BI functions defined.


In [35]:
# ── Calibration functions ────────────────────────────────────────────────
import random

def _dev_str(dev):
    """torch.device → plain string for torch.autocast."""
    return dev.type if isinstance(dev, torch.device) else str(dev).split(':')[0]

def _calib_speech_encoder(model, dev, samples=None, processor_ref=None):
    s_list = samples if samples is not None else calib_samples
    proc   = processor_ref or base_processor
    s      = random.choice(s_list)
    dev_s  = _dev_str(dev)
    inputs = proc(audio=s['audio_array'], src_lang=SRC_LANG,
                  sampling_rate=SAMPLE_RATE, return_tensors='pt').to(dev)
    with torch.autocast(dev_s, dtype=DTYPE):
        model.speech_encoder(
            input_features=inputs.get('input_features', inputs.get('input_values')),
            attention_mask=inputs.get('attention_mask'),
        )
    del inputs

def _calib_text_decoder(model, dev, samples=None, processor_ref=None):
    s_list = samples if samples is not None else calib_samples
    proc   = processor_ref or base_processor
    s      = random.choice(s_list)
    dev_s  = _dev_str(dev)
    inputs = proc(audio=s['audio_array'], src_lang=SRC_LANG,
                  sampling_rate=SAMPLE_RATE, return_tensors='pt').to(dev)
    with torch.autocast(dev_s, dtype=DTYPE), torch.no_grad():
        model.generate(**inputs, tgt_lang=TGT_LANG, max_new_tokens=32,
                       return_intermediate_token_ids=False)
    del inputs

print('Calibration functions ready.')


Calibration functions ready.


In [36]:
# ── Phase 1: Prune speech_encoder 24 → 12 layers ────────────────────────
import copy, functools
import torch.nn as nn

TARGET_SE_LAYERS = 12
model_p1 = None

p1_existing, _ = load_model_from_drive('phase1_speech_pruned')
if p1_existing is not None:
    model_p1 = p1_existing.to(DEVICE)
    print_model_breakdown(model_p1, 'Phase 1 — Loaded from Drive')
else:
    work_model = None
    try:
        print('Phase 1: Computing BI scores for speech_encoder...')
        # base_model is on CPU — deepcopy stays on CPU, then move to GPU
        with cuda_phase('Phase 1 deepcopy+prune'):
            work_model = copy.deepcopy(base_model).to(DEVICE)

            se_layers_path = 'speech_encoder.encoder.layers'
            try:
                se_layers = work_model.speech_encoder.encoder.layers
            except AttributeError:
                se_layers = work_model.speech_encoder.layers
                se_layers_path = 'speech_encoder.layers'

            print(f'  Found {len(se_layers)} speech encoder layers.')

            bi_se = compute_bi_for_submodule(
                work_model, se_layers_path,
                calib_fn=_calib_speech_encoder,
                n_calib=min(8, len(calib_samples))
            )
            print('  BI scores:', [f'{x:.3f}' for x in bi_se])

            keep_idx = select_layers_to_keep(bi_se, TARGET_SE_LAYERS,
                                              always_keep_first=2, always_keep_last=2)
            print(f'  Keeping layers: {keep_idx}')

            parent_path = se_layers_path.rsplit('.', 1)[0]
            parent = functools.reduce(getattr, parent_path.split('.'), work_model)
            parent.layers = prune_layers(se_layers, keep_idx)
            sync_config_from_model(work_model)

            model_p1 = work_model
            work_model = None   # transfer ownership
            save_model_to_drive(model_p1, base_processor, 'phase1_speech_pruned')
            print_model_breakdown(model_p1, 'Phase 1 — Speech Enc Pruned')
    except Exception as e:
        print(f'Phase 1 FAILED: {e}')
        free_memory(work_model)
        work_model = None
        raise
    finally:
        free_memory(work_model)

print_vram('Phase 1 done')


[load] Model phase1_speech_pruned not found in Drive either. Return None.
Phase 1: Computing BI scores for speech_encoder...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 7.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.36 GiB is allocated by PyTorch, and 60.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [24]:
!ls models

phase0_baseline


In [ ]:
# ── Phase 1 benchmark ────────────────────────────────────────────────────
p1_ckpt = load_latest_checkpoint('phase1_benchmark')
if p1_ckpt:
    p1_summary = p1_ckpt
    print(f'Restored P1: ASR-BLEU={p1_summary["asr_bleu"]:.2f}')
else:
    free_mms()
    _, _, p1_summary = benchmark_asr_bleu(
        model_p1, base_processor, eval_samples, label='P1_SE_Pruned')
    save_checkpoint(p1_summary, 'phase1_benchmark')

store_summary(p1_summary)
model_p1 = offload(model_p1, 'p1')


---
# Phase 2 — Text Decoder Depth Pruning (ShortGPT BI)
Target: 24 → 12 decoder layers  (~−260M params)

**Why text_decoder matters for S2ST:**  
The text_decoder is the autoregressive first-pass of UnitY2. It generates the intermediate text in the target language, which is then fed into the T2U model. Poor text_decoder output = broken T2U input = garbled audio.


In [ ]:
# ── Phase 2: Prune text_decoder 24 → 12 layers ──────────────────────────
import copy, functools
import torch.nn as nn

TARGET_TD_LAYERS = 12
model_p2 = None

p2_existing, _ = load_model_from_drive('phase2_decoder_pruned')
if p2_existing is not None:
    model_p2 = p2_existing.to(DEVICE)
    print_model_breakdown(model_p2, 'Phase 2 — Loaded from Drive')
else:
    work_model = None
    try:
        with cuda_phase('Phase 2 deepcopy+prune'):
            # model_p1 is on CPU after Phase 1 benchmark
            work_model = copy.deepcopy(model_p1).to(DEVICE)

            try:
                td_layers = work_model.text_decoder.layers
                td_path   = 'text_decoder.layers'
            except AttributeError:
                td_layers = work_model.text_decoder.model.decoder.layers
                td_path   = 'text_decoder.model.decoder.layers'

            print(f'  Found {len(td_layers)} text decoder layers.')

            bi_td = compute_bi_for_submodule(
                work_model, td_path,
                calib_fn=_calib_text_decoder,
                n_calib=min(8, len(calib_samples))
            )
            print('  BI scores:', [f'{x:.3f}' for x in bi_td])

            keep_idx = select_layers_to_keep(bi_td, TARGET_TD_LAYERS,
                                              always_keep_first=2, always_keep_last=2)
            print(f'  Keeping layers: {keep_idx}')

            td_parent = functools.reduce(
                getattr, td_path.rsplit('.layers', 1)[0].split('.'), work_model)
            td_parent.layers = prune_layers(td_layers, keep_idx)
            sync_config_from_model(work_model)

            model_p2 = work_model
            work_model = None
            save_model_to_drive(model_p2, base_processor, 'phase2_decoder_pruned')
            print_model_breakdown(model_p2, 'Phase 2 — Decoder Pruned')
    except Exception as e:
        print(f'Phase 2 FAILED: {e}')
        free_memory(work_model)
        raise
    finally:
        free_memory(work_model)

print_vram('Phase 2 done')


In [ ]:
# ── Phase 2 benchmark ────────────────────────────────────────────────────
p2_ckpt = load_latest_checkpoint('phase2_benchmark')
if p2_ckpt:
    p2_summary = p2_ckpt
    print(f'Restored P2: ASR-BLEU={p2_summary["asr_bleu"]:.2f}')
else:
    free_mms()
    _, _, p2_summary = benchmark_asr_bleu(
        model_p2, base_processor, eval_samples, label='P2_Dec_Pruned')
    save_checkpoint(p2_summary, 'phase2_benchmark')

store_summary(p2_summary)
model_p2 = offload(model_p2, 'p2')


---
# Phase 3 — Width Pruning (Minitron Activation-Based Importance)
Target: prune FFN hidden dim and attention heads in text_decoder  (~−450M params total)

**Minitron method (NeurIPS 2024):**  
1. Run calibration data through the model, collect mean |activation| per FFN neuron and per attention head.  
2. Rank and remove the lowest-importance neurons/heads.  
3. **Weight reconstruction**: after pruning, multiply the remaining weight rows/cols by a correction factor so the expected output stays close to pre-pruning values. This is a simple least-squares approximation—no retraining required here.


In [ ]:
# ── Minitron-style width pruning ─────────────────────────────────────────
import torch.nn as nn

@torch.no_grad()
def collect_ffn_importance(model, layers, calib_fn, n_calib=8, device='cuda'):
    hooks = []
    # Store only the mean per neuron (not full tensors) to save RAM
    act_sums  = {i: None for i in range(len(layers))}
    act_counts = {i: 0   for i in range(len(layers))}

    def make_hook(i):
        def hook(mod, inp, out):
            # out: (B, T, ffn_dim) — reduce immediately, don't store raw
            val = out.detach().float().abs().mean(dim=(0, 1)).cpu()  # (ffn_dim,)
            if act_sums[i] is None:
                act_sums[i] = val
            else:
                act_sums[i] += val
            act_counts[i] += 1
        return hook

    for i, layer in enumerate(layers):
        act_fn = getattr(layer, 'activation_fn', None)
        h_target = act_fn if act_fn is not None else layer.fc1
        hooks.append(h_target.register_forward_hook(make_hook(i)))

    model.eval()
    dev_s = device.type if isinstance(device, torch.device) else str(device).split(':')[0]
    for _ in range(n_calib):
        try:
            calib_fn(model, device)
        except Exception as e:
            print(f'  [FFN calib warn] {e}')
    for h in hooks:
        h.remove()

    importance = []
    for i in range(len(layers)):
        if act_sums[i] is not None and act_counts[i] > 0:
            importance.append(act_sums[i] / act_counts[i])
        else:
            importance.append(torch.ones(layers[i].fc1.out_features))
    return importance

@torch.no_grad()
def prune_ffn_width(layer, importance_vec, keep_ratio=0.5):
    ffn_dim = layer.fc1.out_features
    keep_n  = max(1, int(ffn_dim * keep_ratio))
    top_idx, _ = torch.sort(torch.argsort(importance_vec, descending=True)[:keep_n])

    new_fc1 = nn.Linear(layer.fc1.in_features, keep_n,
                         bias=layer.fc1.bias is not None)
    new_fc1.weight.data = layer.fc1.weight.data[top_idx]
    if layer.fc1.bias is not None:
        new_fc1.bias.data = layer.fc1.bias.data[top_idx]

    new_fc2 = nn.Linear(keep_n, layer.fc2.out_features,
                         bias=layer.fc2.bias is not None)
    new_fc2.weight.data = layer.fc2.weight.data[:, top_idx]
    if layer.fc2.bias is not None:
        new_fc2.bias.data = layer.fc2.bias.data.clone()

    # Delete old weights before assigning new ones
    del layer.fc1, layer.fc2
    layer.fc1, layer.fc2 = new_fc1, new_fc2
    return keep_n

@torch.no_grad()
def collect_head_importance(model, layers, calib_fn, n_calib=8, device='cuda'):
    head_sums   = {i: None for i in range(len(layers))}
    head_counts = {i: 0    for i in range(len(layers))}
    hooks = []

    def make_hook(i):
        def hook(mod, inp, out):
            if isinstance(out, tuple) and len(out) >= 2 and out[1] is not None:
                val = out[1].detach().float().abs().mean(dim=(0, 2, 3)).cpu()
                if head_sums[i] is None:
                    head_sums[i] = val
                else:
                    head_sums[i] += val
                head_counts[i] += 1
        return hook

    for i, layer in enumerate(layers):
        attn = getattr(layer, 'self_attn', None)
        if attn is not None:
            hooks.append(attn.register_forward_hook(make_hook(i)))

    model.eval()
    orig = model.config.output_attentions
    model.config.output_attentions = True
    for _ in range(n_calib):
        try:
            calib_fn(model, device)
        except Exception:
            pass
    model.config.output_attentions = orig
    for h in hooks:
        h.remove()

    importance = []
    for i in range(len(layers)):
        if head_sums[i] is not None and head_counts[i] > 0:
            importance.append(head_sums[i] / head_counts[i])
        else:
            n_h = getattr(layers[i].self_attn, 'num_heads', 16)
            importance.append(torch.ones(n_h))
    return importance

@torch.no_grad()
def prune_attention_heads(layer, head_importance, keep_ratio=0.75):
    attn = layer.self_attn
    n_heads  = attn.num_heads
    head_dim = attn.head_dim
    keep_n   = max(1, int(n_heads * keep_ratio))
    top_idx, _ = torch.sort(torch.argsort(head_importance, descending=True)[:keep_n])

    def slice_out(proj, hidx, hdim):
        idx = torch.cat([torch.arange(h*hdim, (h+1)*hdim) for h in hidx])
        p = nn.Linear(proj.in_features, len(idx), bias=proj.bias is not None)
        p.weight.data = proj.weight.data[idx]
        if proj.bias is not None:
            p.bias.data = proj.bias.data[idx]
        return p

    def slice_in(proj, hidx, hdim):
        idx = torch.cat([torch.arange(h*hdim, (h+1)*hdim) for h in hidx])
        p = nn.Linear(len(idx), proj.out_features, bias=proj.bias is not None)
        p.weight.data = proj.weight.data[:, idx]
        if proj.bias is not None:
            p.bias.data = proj.bias.data.clone()
        return p

    try:
        new_q = slice_out(attn.q_proj, top_idx, head_dim)
        new_k = slice_out(attn.k_proj, top_idx, head_dim)
        new_v = slice_out(attn.v_proj, top_idx, head_dim)
        new_o = slice_in( attn.out_proj, top_idx, head_dim)
        del attn.q_proj, attn.k_proj, attn.v_proj, attn.out_proj
        attn.q_proj, attn.k_proj, attn.v_proj, attn.out_proj = new_q, new_k, new_v, new_o
        attn.num_heads = keep_n
    except Exception as e:
        print(f'  [head prune skip] {e}')
    return keep_n

print('Minitron width-pruning functions defined.')


In [ ]:
# ── Phase 3: Width pruning on text_decoder ───────────────────────────────
import copy

FFN_KEEP_RATIO  = 0.56
HEAD_KEEP_RATIO = 0.75
model_p3 = None

p3_existing, _ = load_model_from_drive('phase3_width_pruned')
if p3_existing is not None:
    model_p3 = p3_existing.to(DEVICE)
    print_model_breakdown(model_p3, 'Phase 3 — Loaded from Drive')
else:
    work_model = None
    try:
        with cuda_phase('Phase 3 width pruning'):
            work_model = copy.deepcopy(model_p2).to(DEVICE)

            try:
                td_layers = list(work_model.text_decoder.layers)
            except AttributeError:
                td_layers = list(work_model.text_decoder.model.decoder.layers)

            print('  Collecting FFN importance...')
            ffn_imp = collect_ffn_importance(
                work_model, td_layers, calib_fn=_calib_text_decoder,
                n_calib=min(8, len(calib_samples)), device=DEVICE)
            for i, (layer, imp) in enumerate(zip(td_layers, ffn_imp)):
                kept = prune_ffn_width(layer, imp, keep_ratio=FFN_KEEP_RATIO)
                if (i + 1) % 4 == 0:
                    print(f'  Layer {i}: FFN {imp.shape[0]} → {kept}')
            gc.collect(); torch.cuda.empty_cache()

            print('  Collecting head importance...')
            head_imp = collect_head_importance(
                work_model, td_layers, calib_fn=_calib_text_decoder,
                n_calib=min(8, len(calib_samples)), device=DEVICE)
            for i, (layer, imp) in enumerate(zip(td_layers, head_imp)):
                kept = prune_attention_heads(layer, imp, keep_ratio=HEAD_KEEP_RATIO)
                if (i + 1) % 4 == 0:
                    print(f'  Layer {i}: heads → {kept}')
            gc.collect(); torch.cuda.empty_cache()

            sync_config_from_model(work_model)
            sample_layer = td_layers[0]
            work_model.config.decoder_ffn_dim = sample_layer.fc1.out_features
            work_model.config.decoder_attention_heads = sample_layer.self_attn.num_heads

            model_p3 = work_model
            work_model = None
            save_model_to_drive(model_p3, base_processor, 'phase3_width_pruned')
            print_model_breakdown(model_p3, 'Phase 3 — Width Pruned')
    except Exception as e:
        print(f'Phase 3 FAILED: {e}')
        free_memory(work_model)
        raise
    finally:
        free_memory(work_model)

print_vram('Phase 3 done')


In [ ]:
p3_ckpt = load_latest_checkpoint('phase3_benchmark')
if p3_ckpt:
    p3_summary = p3_ckpt
    print(f'Restored P3: ASR-BLEU={p3_summary["asr_bleu"]:.2f}')
else:
    free_mms()
    _, _, p3_summary = benchmark_asr_bleu(
        model_p3, base_processor, eval_samples, label='P3_Width_Pruned')
    save_checkpoint(p3_summary, 'phase3_benchmark')

store_summary(p3_summary)
model_p3 = offload(model_p3, 'p3')


---
# Phase 4 — T2U Depth Pruning (ShortGPT BI)
Target: T2U encoder 6→4 layers, T2U decoder 6→4 layers  (~−90M params)

**Critical caution:** The T2U model is non-autoregressive (NAR) and its decoder has a
duration predictor that drives the character-to-unit upsampling chain.  
We prune conservatively (remove only 2 of 6 layers each) and **always keep** the first
and last layers where the duration heads reside.


In [ ]:
def _calib_t2u(model, dev, samples=None, processor_ref=None):
    s_list = samples if samples is not None else calib_samples
    proc   = processor_ref or base_processor
    s      = random.choice(s_list)
    dev_s  = _dev_str(dev)
    inputs = proc(audio=s['audio_array'], src_lang=SRC_LANG,
                  sampling_rate=SAMPLE_RATE, return_tensors='pt').to(dev)
    with torch.autocast(dev_s, dtype=DTYPE), torch.no_grad():
        model.generate(**inputs, tgt_lang=TGT_LANG, max_new_tokens=32,
                       return_intermediate_token_ids=False)
    del inputs

print('T2U calibration function ready.')


In [ ]:
# ── Phase 4: T2U Depth Pruning ───────────────────────────────────────────
import copy, functools
import torch.nn as nn

T2U_ENC_TARGET = 4
T2U_DEC_TARGET = 4
model_p4 = None

p4_existing, _ = load_model_from_drive('phase4_t2u_pruned')
if p4_existing is not None:
    model_p4 = p4_existing.to(DEVICE)
    print_model_breakdown(model_p4, 'Phase 4 — Loaded from Drive')
else:
    work_model = None
    try:
        with cuda_phase('Phase 4 T2U pruning'):
            work_model = copy.deepcopy(model_p3).to(DEVICE)
            t2u   = work_model.t2u_model
            t2u_m = getattr(t2u, 'model', t2u)
            t2u_enc = getattr(t2u_m, 'encoder', None)
            t2u_dec = getattr(t2u_m, 'decoder', None)
            cos = torch.nn.CosineSimilarity(dim=-1)

            for part_name, part, target in [
                ('t2u_encoder', t2u_enc, T2U_ENC_TARGET),
                ('t2u_decoder', t2u_dec, T2U_DEC_TARGET),
            ]:
                if part is None or not hasattr(part, 'layers'):
                    print(f'  {part_name}: layers not found, skipping.')
                    continue
                layers = part.layers
                n = len(layers)
                if n <= target:
                    print(f'  {part_name}: {n} layers, already ≤ target, skip.')
                    continue
                print(f'  {part_name}: {n} → {target} layers')

                # Mean-pool hooks so variable-length T2U seqs don't crash cat()
                h_in  = {i: [] for i in range(n)}
                h_out = {i: [] for i in range(n)}
                hooks = []

                def make_pre(i):
                    def hook(mod, inp):
                        x = inp[0] if isinstance(inp, tuple) else inp
                        h_in[i].append(x.detach().float().mean(dim=1, keepdim=True).cpu())
                    return hook

                def make_post(i):
                    def hook(mod, inp, out):
                        o = out[0] if isinstance(out, (tuple, list)) else out
                        h_out[i].append(o.detach().float().mean(dim=1, keepdim=True).cpu())
                    return hook

                for i, layer in enumerate(layers):
                    hooks.append(layer.register_forward_pre_hook(make_pre(i)))
                    hooks.append(layer.register_forward_hook(make_post(i)))

                n_calib = min(8, len(calib_samples))
                for _ in range(n_calib):
                    try:
                        _calib_t2u(work_model, DEVICE)
                    except Exception:
                        pass

                for h in hooks:
                    h.remove()

                bi_scores = []
                for i in range(n):
                    if h_in[i] and h_out[i]:
                        ic = torch.cat(h_in[i],  dim=0).float()
                        oc = torch.cat(h_out[i], dim=0).float()
                        sim = cos(ic.reshape(-1, ic.size(-1)),
                                  oc.reshape(-1, oc.size(-1))).mean().item()
                        bi_scores.append(1.0 - sim)
                    else:
                        bi_scores.append(1.0)

                print(f'  BI: {[f"{x:.3f}" for x in bi_scores]}')
                keep_idx = select_layers_to_keep(bi_scores, target,
                                                  always_keep_first=1, always_keep_last=1)
                print(f'  Keep: {keep_idx}')
                part.layers = prune_layers(layers, keep_idx)
                del h_in, h_out   # free hook buffers

            sync_config_from_model(work_model)
            model_p4 = work_model
            work_model = None
            save_model_to_drive(model_p4, base_processor, 'phase4_t2u_pruned')
            print_model_breakdown(model_p4, 'Phase 4 — T2U Pruned')
    except Exception as e:
        print(f'Phase 4 FAILED: {e}')
        free_memory(work_model)
        raise
    finally:
        free_memory(work_model)

print_vram('Phase 4 done')


In [ ]:
p4_ckpt = load_latest_checkpoint('phase4_benchmark')
if p4_ckpt:
    p4_summary = p4_ckpt
    print(f'Restored P4: ASR-BLEU={p4_summary["asr_bleu"]:.2f}')
else:
    free_mms()
    _, _, p4_summary = benchmark_asr_bleu(
        model_p4, base_processor, eval_samples, label='P4_T2U_Pruned')
    save_checkpoint(p4_summary, 'phase4_benchmark')

store_summary(p4_summary)
model_p4 = offload(model_p4, 'p4')


---
# Phase 5 — LoRA Fine-Tuning (S2ST Recovery)
After 3 rounds of pruning the model is smaller but needs calibration.
We use **LoRA** (ICLR 2022) with **logit-distillation** from the baseline model
as our recovery recipe—this is the same strategy as Minitron (NeurIPS 2024).

**Setup:**
- LoRA adapters on text_decoder Q/K/V/O projections and T2U encoder self-attention
- Rank r=16, alpha=32
- Loss = CTC-free cross-entropy on text decoder outputs (S2TT supervision)  
  + KL-divergence distillation from frozen baseline text_decoder logits
- Train for 500 steps, batch size 1, gradient accumulation 4
- LR = 3e-4 with cosine schedule


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F


def attach_lora(model, r=16, lora_alpha=32, target_modules=None):
    """
    Attach LoRA adapters to the text_decoder and t2u_model attention layers.
    """
    if target_modules is None:
        target_modules = [
            # text decoder
            'text_decoder.layers.\.\d+\.self_attn.q_proj',
            'text_decoder.layers.\.\d+\.self_attn.k_proj',
            'text_decoder.layers.\.\d+\.self_attn.v_proj',
            'text_decoder.layers.\.\d+\.self_attn.out_proj',
        ]
    # PEFT needs module name patterns without regex — use simple list
    target_simple = ['q_proj', 'k_proj', 'v_proj', 'out_proj']

    lora_cfg = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=target_simple,
        lora_dropout=0.05,
        bias='none',
    )
    # PEFT wraps the full model
    peft_model = get_peft_model(model, lora_cfg)
    peft_model.print_trainable_parameters()
    return peft_model


def make_training_batch(samples, processor, max_dur_s=10.0, device='cuda'):
    """
    Build a training batch for S2TT supervision.
    Returns dict with input_features and decoder_input_ids.
    """
    s = random.choice(samples)
    arr = s['audio_array']
    # Truncate long audio
    max_samples = int(max_dur_s * SAMPLE_RATE)
    arr = arr[:max_samples]

    inputs = processor(
        audio=arr,
        src_lang=SRC_LANG,
        sampling_rate=SAMPLE_RATE,
        return_tensors='pt'
    ).to(device)

    # Encode reference text as decoder labels
    if s['reference_text']:
        text_ids = processor.tokenizer(
            s['reference_text'],
            return_tensors='pt',
            max_length=128,
            truncation=True,
            padding=True
        ).input_ids.to(device)
    else:
        text_ids = None

    inputs['labels'] = text_ids
    return inputs

print('LoRA fine-tuning functions ready.')

In [ ]:
# ── Phase 5: LoRA fine-tuning ────────────────────────────────────────────
import copy

MAX_TRAIN_STEPS = 200   # reduced from 500 for faster walkthrough
GRAD_ACCUM      = 4
LR              = 3e-4
KD_ALPHA        = 0.5
KD_TEMPERATURE  = 2.0
model_p5 = None

p5_existing, _ = load_model_from_drive('phase5_lora_trained')
if p5_existing is not None:
    model_p5 = p5_existing.to(DEVICE)
    print_model_breakdown(model_p5, 'Phase 5 — Loaded from Drive')
else:
    student, teacher, optimizer, scheduler, scaler = None, None, None, None, None
    try:
        print('Phase 5: LoRA fine-tuning...')
        free_mms()   # free MMS before loading two 2B models

        student = copy.deepcopy(model_p4).to(DEVICE)
        student = attach_lora(student, r=16, lora_alpha=32)
        student.train()

        # Teacher on CPU to avoid OOM — forward in no_grad is fine
        teacher = base_model   # already on CPU
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)

        optimizer = AdamW(
            [p for p in student.parameters() if p.requires_grad],
            lr=LR, weight_decay=0.01)
        scheduler = CosineAnnealingLR(optimizer, T_max=MAX_TRAIN_STEPS)
        scaler    = torch.cuda.amp.GradScaler(enabled=(DTYPE == torch.float16))

        losses, step = [], 0
        optimizer.zero_grad()
        train_samples = calib_samples

        while step < MAX_TRAIN_STEPS:
            batch  = make_training_batch(train_samples, base_processor, device=DEVICE)
            labels = batch.pop('labels', None)
            if labels is None:
                step += 1
                continue

            try:
                with torch.autocast(DEVICE, dtype=DTYPE):
                    s_out = student(**{k: v for k, v in batch.items()},
                                   decoder_input_ids=labels,
                                   output_hidden_states=False)
                    s_logits = s_out.logits if hasattr(s_out, 'logits') else s_out[0]

                    shift_logits = s_logits[:, :-1, :].contiguous()
                    shift_labels = labels[:, 1:].contiguous()
                    ce_loss = F.cross_entropy(
                        shift_logits.view(-1, shift_logits.size(-1)),
                        shift_labels.view(-1), ignore_index=-100)

                    # Teacher on CPU → move batch to CPU just for teacher fwd
                    with torch.no_grad():
                        cpu_batch = {k: v.cpu() for k, v in batch.items()}
                        t_out = teacher(**cpu_batch,
                                        decoder_input_ids=labels.cpu())
                        t_logits = (t_out.logits if hasattr(t_out, 'logits')
                                    else t_out[0]).to(DEVICE)
                        del cpu_batch

                    T = KD_TEMPERATURE
                    kd_loss = F.kl_div(
                        F.log_softmax(shift_logits / T, dim=-1),
                        F.softmax(t_logits[:, :-1, :].contiguous() / T, dim=-1),
                        reduction='batchmean') * (T ** 2)

                    loss = ((1 - KD_ALPHA) * ce_loss + KD_ALPHA * kd_loss) / GRAD_ACCUM

                scaler.scale(loss).backward()

                if (step + 1) % GRAD_ACCUM == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        [p for p in student.parameters() if p.requires_grad], 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

                losses.append(loss.item() * GRAD_ACCUM)

            except Exception as e:
                print(f'  [step {step}] error: {e}; skipping')
                optimizer.zero_grad()
            finally:
                del batch, labels
                torch.cuda.empty_cache()

            step += 1
            if step % 50 == 0:
                avg = float(np.mean(losses[-50:]))
                print(f'  Step {step:3d}/{MAX_TRAIN_STEPS}  loss={avg:.4f}  '
                      f'lr={scheduler.get_last_lr()[0]:.2e}')
                save_checkpoint({'step': step, 'loss': losses},
                                'phase5_training', step=step)
                print_vram('training')

        print('Merging LoRA adapters...')
        merged = student.merge_and_unload()

        # Teardown training objects
        free_memory(student)
        student = None
        del optimizer, scheduler, scaler
        gc.collect(); torch.cuda.empty_cache()

        model_p5 = merged
        save_model_to_drive(model_p5, base_processor, 'phase5_lora_trained')
        print_model_breakdown(model_p5, 'Phase 5 — LoRA Merged')

        plt.figure(figsize=(10, 4))
        plt.plot(losses)
        plt.xlabel('Step'); plt.ylabel('Loss')
        plt.title('Phase 5 LoRA Training Loss')
        plt.savefig(f'{FIG_DIR}/phase5_loss.png', dpi=100)
        plt.close()

    except Exception as e:
        print(f'Phase 5 FAILED: {e}')
        free_memory(student)
        raise
    finally:
        free_memory(student)
        del optimizer, scheduler, scaler
        gc.collect(); torch.cuda.empty_cache()

print_vram('Phase 5 done')


In [ ]:
p5_ckpt = load_latest_checkpoint('phase5_benchmark')
if p5_ckpt:
    p5_summary = p5_ckpt
    print(f'Restored P5: ASR-BLEU={p5_summary["asr_bleu"]:.2f}')
else:
    free_mms()
    _, _, p5_summary = benchmark_asr_bleu(
        model_p5, base_processor, eval_samples, label='P5_LoRA_Recovery')
    save_checkpoint(p5_summary, 'phase5_benchmark')

store_summary(p5_summary)
model_p5 = offload(model_p5, 'p5')


---
# Phase 6 — Final Results Table


In [ ]:
# ── Reload all summaries ──────────────────────────────────────────────────
sc = load_latest_checkpoint('all_summaries')
if sc and 'summaries' in sc:
    ALL_SUMMARIES = sc['summaries']

print('\n' + '='*75)
print('  SeamlessM4T v2 Large  —  S2ST Compression Pipeline (eng→ben)')
print('  Metric: ASR-BLEU (Whisper-medium transcription + sacrebleu)')
print('='*75)
hdr = f'{"Phase":<28} {"Params (M)":>10} {"Δ Size":>8} {"ASR-BLEU":>9} {"RTF":>7}'
print(hdr)
print('─'*len(hdr))
bp = ALL_SUMMARIES[0]['params_M'] if ALL_SUMMARIES else 2300
for s in ALL_SUMMARIES:
    d = (1 - s['params_M'] / bp) * 100 if bp else 0
    ds = f'-{d:.1f}%' if d > 0.1 else 'base'
    print(f'  {s["label"]:<26} {s["params_M"]:>8.0f}  '
          f'{ds:>7}  {s["asr_bleu"]:>8.2f}  {s["avg_rtf"]:>6.3f}')
print('='*75)
if len(ALL_SUMMARIES) >= 2:
    first, last = ALL_SUMMARIES[0], ALL_SUMMARIES[-1]
    cr = (1 - last['params_M'] / first['params_M']) * 100
    qr = (last['asr_bleu'] / max(first['asr_bleu'], 1e-6)) * 100
    print(f'\n  Compression ratio : {cr:.1f}% fewer parameters')
    print(f'  Quality retention : {qr:.1f}% of baseline ASR-BLEU')
    if last['avg_rtf'] > 0 and first['avg_rtf'] > 0:
        print(f'  Speed-up (RTF)    : {first["avg_rtf"]/last["avg_rtf"]:.2f}x faster')

In [ ]:
if len(ALL_SUMMARIES) >= 2:
    labels  = [s['label'] for s in ALL_SUMMARIES]
    params  = [s['params_M'] for s in ALL_SUMMARIES]
    bleus   = [s['asr_bleu'] for s in ALL_SUMMARIES]
    rtfs    = [s['avg_rtf'] for s in ALL_SUMMARIES]
    x       = range(len(labels))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('SeamlessM4T v2 → ~1B  Compression Results', fontsize=14, fontweight='bold')

    ax = axes[0]
    bars = ax.bar(x, params, color='#7C3AED', alpha=0.85, edgecolor='white')
    ax.set_ylabel('Parameters (M)')
    ax.set_title('Model Size', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.axhline(1000, color='red', ls='--', lw=1.5, label='1B target')
    ax.legend(fontsize=8)

    ax = axes[1]
    ax.plot(x, bleus, 'o-', color='#0EA5E9', lw=2, ms=7)
    ax.fill_between(x, [b * 0.9 for b in bleus], bleus, alpha=0.15, color='#0EA5E9')
    ax.set_ylabel('ASR-BLEU')
    ax.set_title('ASR-BLEU (higher = better)', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)

    ax = axes[2]
    base_b = bleus[0] or 1
    base_p = params[0] or 1
    comp   = [(1 - p/base_p)*100 for p in params]
    qual   = [b/base_b*100 for b in bleus]
    sc_plt = ax.scatter(comp, qual, c=range(len(labels)),
                        cmap='plasma', s=120, zorder=5)
    for i, lbl in enumerate(labels):
        ax.annotate(lbl, (comp[i], qual[i]), fontsize=7, ha='left', va='bottom')
    ax.axhline(90, color='green', ls='--', lw=1, label='90% quality')
    ax.axvline(55, color='red',   ls='--', lw=1, label='55% compression')
    ax.set_xlabel('Compression (%)')
    ax.set_ylabel('ASR-BLEU retention (%)')
    ax.set_title('Compression vs Quality', fontweight='bold')
    ax.legend(fontsize=7)

    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/compression_results.png', dpi=150)
    plt.show()
    _rclone_push(f'{FIG_DIR}/compression_results.png',
                 'figures/compression_results.png')
    print('Figure saved.')

In [ ]:
print('Done! Final compressed model: phase5_lora_trained')
if DEVICE == 'cuda':
    print(f'Peak VRAM used: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
print(f'Work dir: {WORK_DIR}')
print(f'Models  : {MODEL_DIR}')